# Parallel sorting (chapter 14)

Based on chapter 14 of *Programming Massively Parallel Processors* (Hwu, Kirk, El Hajj). Sorting is where several earlier patterns meet. The parallel **radix sort** of this chapter is built from a map (extract a digit), an exclusive **scan** (chapter 11) to compute where every element goes, and a scatter. Its optimized version adds tiling in shared memory and a small table of bucket counts. The parallel **merge sort** is built from the co-rank merge of chapter 13.

This notebook goes **step by step**: 5 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/12_sorting/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 5: `01_radix_pass_sequential.cu`

One radix pass done sequentially by a single GPU thread: count the zeros, then move the keys with bit 0 to the front and the keys with bit 1 after them, in input order. Each key is printed with its original position in parentheses, so stability is visible.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_radix_pass_sequential.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <vector>

constexpr int N = 8;
constexpr int BIT = 0;

__host__ __device__ unsigned bit_of(unsigned key, int bit) {
  return (key >> bit) & 1u;
}

__host__ __device__ void radix_pass_sequential(const unsigned* keysIn,
                                               const unsigned* valsIn,
                                               int n,
                                               int bit,
                                               unsigned* keysOut,
                                               unsigned* valsOut) {
  unsigned zeros = 0;
  for (int i = 0; i < n; ++i) {
    zeros += 1u - bit_of(keysIn[i], bit);
  }

  unsigned zerosBefore = 0;
  unsigned onesBefore = 0;
  for (int i = 0; i < n; ++i) {
    const bool isOne = bit_of(keysIn[i], bit) == 1u;
    const unsigned dst = isOne ? zeros + onesBefore : zerosBefore;
    keysOut[dst] = keysIn[i];
    valsOut[dst] = valsIn[i];
    onesBefore += isOne ? 1u : 0u;
    zerosBefore += isOne ? 0u : 1u;
  }
}

__global__ void radix_pass_sequential_kernel(const unsigned* keysIn,
                                             const unsigned* valsIn,
                                             int n,
                                             int bit,
                                             unsigned* keysOut,
                                             unsigned* valsOut) {
  radix_pass_sequential(keysIn, valsIn, n, bit, keysOut, valsOut);
}

struct Pair {
  unsigned key;
  unsigned val;
};

static unsigned* toDevice(const std::vector<unsigned>& values) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(unsigned)));
  CHECK(
      cudaMemcpy(pointer, values.data(), values.size() * sizeof(unsigned), cudaMemcpyHostToDevice));
  return pointer;
}

static std::vector<unsigned> toHost(const unsigned* pointer, int count) {
  std::vector<unsigned> values(count);
  CHECK(cudaMemcpy(values.data(), pointer, count * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return values;
}

static void printPairs(const char* label,
                       const std::vector<unsigned>& keys,
                       const std::vector<unsigned>& vals) {
  std::printf("%-14s", label);
  for (int i = 0; i < N; ++i) {
    std::printf("  %u(%u)", keys[i], vals[i]);
  }
  std::printf("\n");
}

static std::vector<Pair> stablePartitionOnHost(const std::vector<unsigned>& keys,
                                               const std::vector<unsigned>& vals) {
  std::vector<Pair> pairs(N);
  for (int i = 0; i < N; ++i) {
    pairs[i] = {keys[i], vals[i]};
  }
  std::stable_partition(pairs.begin(), pairs.end(), [](const Pair& pair) {
    return bit_of(pair.key, BIT) == 0u;
  });
  return pairs;
}

int main() {
  const std::vector<unsigned> keys = {5, 2, 7, 2, 4, 1, 6, 3};
  const std::vector<unsigned> vals = {0, 1, 2, 3, 4, 5, 6, 7};
  unsigned* keysIn_d = toDevice(keys);
  unsigned* valsIn_d = toDevice(vals);
  unsigned* keysOut_d = toDevice(keys);
  unsigned* valsOut_d = toDevice(vals);

  launch(radix_pass_sequential_kernel,
         dim3(1),
         dim3(1),
         keysIn_d,
         valsIn_d,
         N,
         BIT,
         keysOut_d,
         valsOut_d);
  CHECK(cudaGetLastError());
  const std::vector<unsigned> keysOut = toHost(keysOut_d, N);
  const std::vector<unsigned> valsOut = toHost(valsOut_d, N);

  printPairs("input:", keys, vals);
  printPairs("after bit 0:", keysOut, valsOut);

  CHECK(cudaFree(keysIn_d));
  CHECK(cudaFree(valsIn_d));
  CHECK(cudaFree(keysOut_d));
  CHECK(cudaFree(valsOut_d));

  const std::vector<Pair> expected = stablePartitionOnHost(keys, vals);
  bool ok = true;
  for (int i = 0; i < N; ++i) {
    ok = ok && keysOut[i] == expected[i].key && valsOut[i] == expected[i].val;
  }
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_radix_pass_sequential.cu -o 01_radix_pass_sequential && ./01_radix_pass_sequential

**What to look at in the output:** Follow the numbers in parentheses: after the pass the even keys 2(1) 2(3) 4(4) 6(6) are at the front and the odd keys 5(0) 7(2) 1(5) 3(7) after them, each group in its original order. The two 2s keep their order, 2(1) before 2(3); that is stability, and the later passes rely on it.

## Step 2 of 5: `02_radix_sort_1bit.cu`

The parallel 1-bit radix sort: a map extracts the bits, an exclusive scan (one block, Kogge-Stone) gives `onesBefore`, and the scatter kernel computes every destination. The bits, the scan and the array are printed after every bit.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_radix_sort_1bit.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <utility>
#include <vector>

constexpr int N = 8;
constexpr int KEY_BITS = 3;

__host__ __device__ unsigned bit_of(unsigned key, int bit) {
  return (key >> bit) & 1u;
}

__global__ void radix_bit_kernel(const unsigned* keys, unsigned* bits, int n, int bit) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    bits[i] = bit_of(keys[i], bit);
  }
}

__global__ void exclusive_scan_kernel(const unsigned* in, unsigned* out, int n) {
  __shared__ unsigned buffer[2][N];
  const int t = threadIdx.x;
  buffer[0][t] = t > 0 && t < n ? in[t - 1] : 0u;
  __syncthreads();

  int readHalf = 0;
  for (int stride = 1; stride < N; stride *= 2) {
    const int writeHalf = 1 - readHalf;
    unsigned sum = buffer[readHalf][t];
    if (t >= stride) {
      sum += buffer[readHalf][t - stride];
    }
    buffer[writeHalf][t] = sum;
    readHalf = writeHalf;
    __syncthreads();
  }

  if (t < n) {
    out[t] = buffer[readHalf][t];
  }
}

__global__ void radix_scatter_kernel(const unsigned* keysIn,
                                     const unsigned* valsIn,
                                     const unsigned* bits,
                                     const unsigned* onesBefore,
                                     int n,
                                     unsigned* keysOut,
                                     unsigned* valsOut) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    const unsigned totalOnes = onesBefore[n - 1] + bits[n - 1];
    const unsigned zeros = n - totalOnes;
    const unsigned dst = bits[i] ? zeros + onesBefore[i] : i - onesBefore[i];
    keysOut[dst] = keysIn[i];
    valsOut[dst] = valsIn[i];
  }
}

struct Pair {
  unsigned key;
  unsigned val;
};

static unsigned* toDevice(const std::vector<unsigned>& values) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(unsigned)));
  CHECK(
      cudaMemcpy(pointer, values.data(), values.size() * sizeof(unsigned), cudaMemcpyHostToDevice));
  return pointer;
}

static std::vector<unsigned> toHost(const unsigned* pointer, int count) {
  std::vector<unsigned> values(count);
  CHECK(cudaMemcpy(values.data(), pointer, count * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return values;
}

static void printValues(const char* label, const std::vector<unsigned>& values) {
  std::printf("%-14s", label);
  for (unsigned value : values) {
    std::printf("  %4u", value);
  }
  std::printf("\n");
}

static void printPairs(const char* label,
                       const std::vector<unsigned>& keys,
                       const std::vector<unsigned>& vals) {
  std::printf("%-14s", label);
  for (int i = 0; i < N; ++i) {
    std::printf("  %u(%u)", keys[i], vals[i]);
  }
  std::printf("\n");
}

static void radixPass(
    unsigned* keys_d[2], unsigned* vals_d[2], unsigned* bits_d, unsigned* onesBefore_d, int bit) {
  launch(radix_bit_kernel, dim3(1), dim3(N), keys_d[0], bits_d, N, bit);
  launch(exclusive_scan_kernel, dim3(1), dim3(N), bits_d, onesBefore_d, N);
  launch(radix_scatter_kernel,
         dim3(1),
         dim3(N),
         keys_d[0],
         vals_d[0],
         bits_d,
         onesBefore_d,
         N,
         keys_d[1],
         vals_d[1]);
  CHECK(cudaGetLastError());
  std::swap(keys_d[0], keys_d[1]);
  std::swap(vals_d[0], vals_d[1]);
}

int main() {
  const std::vector<unsigned> keys = {5, 2, 7, 2, 4, 1, 6, 3};
  const std::vector<unsigned> vals = {0, 1, 2, 3, 4, 5, 6, 7};
  unsigned* keys_d[2] = {toDevice(keys), toDevice(keys)};
  unsigned* vals_d[2] = {toDevice(vals), toDevice(vals)};
  unsigned* bits_d = toDevice(keys);
  unsigned* onesBefore_d = toDevice(keys);

  printPairs("input:", keys, vals);
  for (int bit = 0; bit < KEY_BITS; ++bit) {
    radixPass(keys_d, vals_d, bits_d, onesBefore_d, bit);
    std::printf("\nbit %d\n", bit);
    printValues("bits:", toHost(bits_d, N));
    printValues("onesBefore:", toHost(onesBefore_d, N));
    printPairs("after:", toHost(keys_d[0], N), toHost(vals_d[0], N));
  }

  const std::vector<unsigned> sortedKeys = toHost(keys_d[0], N);
  const std::vector<unsigned> sortedVals = toHost(vals_d[0], N);
  for (int buffer = 0; buffer < 2; ++buffer) {
    CHECK(cudaFree(keys_d[buffer]));
    CHECK(cudaFree(vals_d[buffer]));
  }
  CHECK(cudaFree(bits_d));
  CHECK(cudaFree(onesBefore_d));

  std::vector<Pair> expected(N);
  for (int i = 0; i < N; ++i) {
    expected[i] = {keys[i], vals[i]};
  }
  std::stable_sort(expected.begin(), expected.end(), [](const Pair& a, const Pair& b) {
    return a.key < b.key;
  });
  bool ok = true;
  for (int i = 0; i < N; ++i) {
    ok = ok && sortedKeys[i] == expected[i].key && sortedVals[i] == expected[i].val;
  }
  std::printf("\n%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_radix_sort_1bit.cu -o 02_radix_sort_1bit && ./02_radix_sort_1bit

**What to look at in the output:** Check one destination per kind in bit 0: key 5(0) has bit 1 and `onesBefore` 0, so it goes to 4 + 0 = 4, right after the 4 zeros; key 2(3) has bit 0 and `onesBefore` 2, so it goes to 3 − 2 = 1. After bit 1 the keys are sorted by their two lowest bits, and after bit 2 by all three, with 2(1) still before 2(3). Try: set `KEY_BITS = 2`; only two passes run, the result 4 5 1 2 2 6 7 3 is sorted by the lowest two bits only and the check fails.

## Step 3 of 5: `03_radix_local_sort.cu`

Local sort in shared memory: each block splits its tile of 4 keys by the bit with a block scan and writes its bucket counts into a bucket-major table; the table is scanned and the coalesced scatter writes each bucket of each tile as one run (the worked example of the README).

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_radix_local_sort.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <utility>
#include <vector>

constexpr int N = 8;
constexpr int KEY_BITS = 3;
constexpr int TILE = 4;
constexpr int BLOCKS = (N + TILE - 1) / TILE;
constexpr int BUCKETS = 2;
constexpr int TABLE_SIZE = BUCKETS * BLOCKS;
constexpr unsigned PADDING_KEY = 0xFFFFFFFFu;

__host__ __device__ unsigned bit_of(unsigned key, int bit) {
  return (key >> bit) & 1u;
}

__host__ __device__ int bucket_major_index(int bucket, int block, int numBlocks) {
  return bucket * numBlocks + block;
}

__host__ __device__ int block_major_index(int block, int bucket, int buckets) {
  return block * buckets + bucket;
}

__device__ unsigned block_inclusive_scan(unsigned value, unsigned* buf, unsigned* total) {
  const int t = threadIdx.x;
  const int threads = blockDim.x;
  buf[t] = value;
  __syncthreads();

  int readHalf = 0;
  for (int stride = 1; stride < threads; stride *= 2) {
    const int writeHalf = 1 - readHalf;
    unsigned sum = buf[readHalf * threads + t];
    if (t >= stride) {
      sum += buf[readHalf * threads + t - stride];
    }
    buf[writeHalf * threads + t] = sum;
    readHalf = writeHalf;
    __syncthreads();
  }

  *total = buf[readHalf * threads + threads - 1];
  const unsigned inclusivePrefix = buf[readHalf * threads + t];
  __syncthreads();
  return inclusivePrefix;
}

__global__ void radix_local_sort_kernel(const unsigned* keysIn,
                                        int n,
                                        int bit,
                                        unsigned* keysTmp,
                                        unsigned* counts,
                                        unsigned* localStart) {
  __shared__ unsigned keys_S[TILE];
  __shared__ unsigned buf[2 * TILE];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  const bool isRealElement = t < tileCount;
  const unsigned key = isRealElement ? keysIn[tileBase + t] : PADDING_KEY;
  const unsigned myBit = bit_of(key, bit);

  unsigned totalOnes;
  const unsigned onesBefore = block_inclusive_scan(myBit, buf, &totalOnes) - myBit;
  const unsigned zeros = TILE - totalOnes;
  const unsigned dst = myBit ? zeros + onesBefore : t - onesBefore;
  keys_S[dst] = key;
  __syncthreads();

  if (isRealElement) {
    keysTmp[tileBase + t] = keys_S[t];
  }
  if (t == 0) {
    counts[bucket_major_index(0, blockIdx.x, gridDim.x)] = zeros;
    counts[bucket_major_index(1, blockIdx.x, gridDim.x)] = tileCount - zeros;
    localStart[block_major_index(blockIdx.x, 0, BUCKETS)] = 0;
    localStart[block_major_index(blockIdx.x, 1, BUCKETS)] = zeros;
  }
}

__global__ void scan_table_kernel(const unsigned* counts, unsigned* offsets, int size) {
  __shared__ unsigned buf[2 * TABLE_SIZE];
  const int t = threadIdx.x;
  const unsigned count = t < size ? counts[t] : 0u;
  unsigned total;
  const unsigned inclusivePrefix = block_inclusive_scan(count, buf, &total);
  if (t < size) {
    offsets[t] = inclusivePrefix - count;
  }
}

__global__ void radix_scatter_coalesced_kernel(const unsigned* keysTmp,
                                               int n,
                                               int bit,
                                               const unsigned* offsets,
                                               const unsigned* localStart,
                                               unsigned* keysOut) {
  __shared__ int localToGlobal[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int d = t; d < BUCKETS; d += blockDim.x) {
    const int globalStart = offsets[bucket_major_index(d, blockIdx.x, gridDim.x)];
    const int bucketLocalStart = localStart[block_major_index(blockIdx.x, d, BUCKETS)];
    localToGlobal[d] = globalStart - bucketLocalStart;
  }
  __syncthreads();

  if (t < tileCount) {
    const unsigned key = keysTmp[tileBase + t];
    const int dst = localToGlobal[bit_of(key, bit)] + t;
    keysOut[dst] = key;
  }
}

struct SortBuffers {
  unsigned* keys[2];
  unsigned* keysTmp;
  unsigned* counts;
  unsigned* offsets;
  unsigned* localStart;
};

static unsigned* toDevice(const std::vector<unsigned>& values) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(unsigned)));
  CHECK(
      cudaMemcpy(pointer, values.data(), values.size() * sizeof(unsigned), cudaMemcpyHostToDevice));
  return pointer;
}

static std::vector<unsigned> toHost(const unsigned* pointer, int count) {
  std::vector<unsigned> values(count);
  CHECK(cudaMemcpy(values.data(), pointer, count * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return values;
}

static void printValues(const char* label, const std::vector<unsigned>& values) {
  std::printf("%-24s", label);
  for (unsigned value : values) {
    std::printf(" %2u", value);
  }
  std::printf("\n");
}

static void radixPass(SortBuffers& buffers, int bit) {
  launch(radix_local_sort_kernel,
         dim3(BLOCKS),
         dim3(TILE),
         buffers.keys[0],
         N,
         bit,
         buffers.keysTmp,
         buffers.counts,
         buffers.localStart);
  launch(scan_table_kernel, dim3(1), dim3(TABLE_SIZE), buffers.counts, buffers.offsets, TABLE_SIZE);
  launch(radix_scatter_coalesced_kernel,
         dim3(BLOCKS),
         dim3(TILE),
         buffers.keysTmp,
         N,
         bit,
         buffers.offsets,
         buffers.localStart,
         buffers.keys[1]);
  CHECK(cudaGetLastError());
  std::swap(buffers.keys[0], buffers.keys[1]);
}

static void printPass(const SortBuffers& buffers, int bit) {
  std::printf("\nbit %d\n", bit);
  printValues("tiles after local sort:", toHost(buffers.keysTmp, N));
  printValues("counts (bucket-major):", toHost(buffers.counts, TABLE_SIZE));
  printValues("offsets (scan):", toHost(buffers.offsets, TABLE_SIZE));
  printValues("after scatter:", toHost(buffers.keys[0], N));
}

int main() {
  const std::vector<unsigned> keys = {5, 2, 7, 2, 4, 1, 6, 3};
  const std::vector<unsigned> table(TABLE_SIZE, 0u);
  SortBuffers buffers = {{toDevice(keys), toDevice(keys)},
                         toDevice(keys),
                         toDevice(table),
                         toDevice(table),
                         toDevice(table)};

  printValues("input:", keys);
  for (int bit = 0; bit < KEY_BITS; ++bit) {
    radixPass(buffers, bit);
    printPass(buffers, bit);
  }

  const std::vector<unsigned> sortedKeys = toHost(buffers.keys[0], N);
  unsigned* allBuffers[] = {buffers.keys[0],
                            buffers.keys[1],
                            buffers.keysTmp,
                            buffers.counts,
                            buffers.offsets,
                            buffers.localStart};
  for (unsigned* buffer : allBuffers) {
    CHECK(cudaFree(buffer));
  }

  std::vector<unsigned> expected = keys;
  std::sort(expected.begin(), expected.end());
  const bool ok = sortedKeys == expected;
  std::printf("\n%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_radix_local_sort.cu -o 03_radix_local_sort && ./03_radix_local_sort

**What to look at in the output:** In bit 0 each tile of 4 keys is split on its own (5 2 7 2 becomes 2 2 5 7), and the bucket-major table lists the zeros of tile 0, the zeros of tile 1, the ones of tile 0 and the ones of tile 1: 2 2 2 2. Its exclusive scan 0 2 4 6 is where each run starts, so the ones of tile 0 (5 7) land at positions 4 and 5. In bit 1 the counts become 1 2 3 2 and the offsets 0 1 3 6; the final array is the same as in step 2.

## Step 4 of 5: `04_radix_multibit.cu`

Multi-bit radix with 4-bit digits: the local sort does 4 one-bit splits per pass and finds where each of the 16 buckets starts and ends in the tile, so 8-bit keys need 2 passes instead of 8. Keys are printed in hex, one hex digit per pass.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_radix_multibit.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <utility>
#include <vector>

constexpr int N = 16;
constexpr int KEY_BITS = 8;
constexpr unsigned KEY_RANGE = 1u << KEY_BITS;
constexpr unsigned DISTINCT_KEYS = 12;
constexpr unsigned KEY_STEP = 0x5b;
constexpr unsigned FIRST_KEY = 0x3a;
constexpr int RADIX_BITS = 4;
constexpr int TILE = 8;
constexpr int BLOCKS = (N + TILE - 1) / TILE;
constexpr int BUCKETS = 1 << RADIX_BITS;
constexpr int TABLE_SIZE = BUCKETS * BLOCKS;
constexpr unsigned PADDING_KEY = 0xFFFFFFFFu;

__host__ __device__ unsigned bit_of(unsigned key, int bit) {
  return (key >> bit) & 1u;
}

__host__ __device__ unsigned digit_of(unsigned key, int shift) {
  return (key >> shift) & (BUCKETS - 1u);
}

__host__ __device__ int bucket_major_index(int bucket, int block, int numBlocks) {
  return bucket * numBlocks + block;
}

__host__ __device__ int block_major_index(int block, int bucket, int buckets) {
  return block * buckets + bucket;
}

__device__ unsigned block_inclusive_scan(unsigned value, unsigned* buf, unsigned* total) {
  const int t = threadIdx.x;
  const int threads = blockDim.x;
  buf[t] = value;
  __syncthreads();

  int readHalf = 0;
  for (int stride = 1; stride < threads; stride *= 2) {
    const int writeHalf = 1 - readHalf;
    unsigned sum = buf[readHalf * threads + t];
    if (t >= stride) {
      sum += buf[readHalf * threads + t - stride];
    }
    buf[writeHalf * threads + t] = sum;
    readHalf = writeHalf;
    __syncthreads();
  }

  *total = buf[readHalf * threads + threads - 1];
  const unsigned inclusivePrefix = buf[readHalf * threads + t];
  __syncthreads();
  return inclusivePrefix;
}

__global__ void radix_local_sort_kernel(const unsigned* keysIn,
                                        int n,
                                        int shift,
                                        unsigned* keysTmp,
                                        unsigned* counts,
                                        unsigned* localStart) {
  __shared__ unsigned keys_S[TILE];
  __shared__ unsigned buf[2 * TILE];
  __shared__ unsigned bucketStart[BUCKETS];
  __shared__ unsigned bucketEnd[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int d = t; d < BUCKETS; d += blockDim.x) {
    bucketStart[d] = 0;
    bucketEnd[d] = 0;
  }
  keys_S[t] = t < tileCount ? keysIn[tileBase + t] : PADDING_KEY;
  __syncthreads();

  for (int bitInDigit = 0; bitInDigit < RADIX_BITS; ++bitInDigit) {
    const unsigned key = keys_S[t];
    const unsigned myBit = bit_of(key, shift + bitInDigit);
    unsigned totalOnes;
    const unsigned onesBefore = block_inclusive_scan(myBit, buf, &totalOnes) - myBit;
    const unsigned zeros = TILE - totalOnes;
    const unsigned dst = myBit ? zeros + onesBefore : t - onesBefore;
    keys_S[dst] = key;
    __syncthreads();
  }

  if (t < tileCount) {
    const unsigned d = digit_of(keys_S[t], shift);
    const bool startsBucket = t == 0 || digit_of(keys_S[t - 1], shift) != d;
    const bool endsBucket = t == tileCount - 1 || digit_of(keys_S[t + 1], shift) != d;
    if (startsBucket) {
      bucketStart[d] = t;
    }
    if (endsBucket) {
      bucketEnd[d] = t + 1;
    }
    keysTmp[tileBase + t] = keys_S[t];
  }
  __syncthreads();

  for (int d = t; d < BUCKETS; d += blockDim.x) {
    counts[bucket_major_index(d, blockIdx.x, gridDim.x)] = bucketEnd[d] - bucketStart[d];
    localStart[block_major_index(blockIdx.x, d, BUCKETS)] = bucketStart[d];
  }
}

__global__ void scan_table_kernel(const unsigned* counts, unsigned* offsets, int size) {
  __shared__ unsigned buf[2 * TABLE_SIZE];
  const int t = threadIdx.x;
  const unsigned count = t < size ? counts[t] : 0u;
  unsigned total;
  const unsigned inclusivePrefix = block_inclusive_scan(count, buf, &total);
  if (t < size) {
    offsets[t] = inclusivePrefix - count;
  }
}

__global__ void radix_scatter_coalesced_kernel(const unsigned* keysTmp,
                                               int n,
                                               int shift,
                                               const unsigned* offsets,
                                               const unsigned* localStart,
                                               unsigned* keysOut) {
  __shared__ int localToGlobal[BUCKETS];
  const int t = threadIdx.x;
  const int tileBase = blockIdx.x * TILE;
  const int tileCount = imin(TILE, n - tileBase);

  for (int d = t; d < BUCKETS; d += blockDim.x) {
    const int globalStart = offsets[bucket_major_index(d, blockIdx.x, gridDim.x)];
    const int bucketLocalStart = localStart[block_major_index(blockIdx.x, d, BUCKETS)];
    localToGlobal[d] = globalStart - bucketLocalStart;
  }
  __syncthreads();

  if (t < tileCount) {
    const unsigned key = keysTmp[tileBase + t];
    const int dst = localToGlobal[digit_of(key, shift)] + t;
    keysOut[dst] = key;
  }
}

struct SortBuffers {
  unsigned* keys[2];
  unsigned* keysTmp;
  unsigned* counts;
  unsigned* offsets;
  unsigned* localStart;
};

static unsigned* toDevice(const std::vector<unsigned>& values) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(unsigned)));
  CHECK(
      cudaMemcpy(pointer, values.data(), values.size() * sizeof(unsigned), cudaMemcpyHostToDevice));
  return pointer;
}

static std::vector<unsigned> toHost(const unsigned* pointer, int count) {
  std::vector<unsigned> values(count);
  CHECK(cudaMemcpy(values.data(), pointer, count * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return values;
}

static void printKeys(const char* label, const std::vector<unsigned>& keys) {
  std::printf("%-34s", label);
  for (unsigned key : keys) {
    std::printf(" %02x", key);
  }
  std::printf("\n");
}

static void printCounts(const std::vector<unsigned>& counts, int block) {
  std::printf("counts of block %d, buckets 0..f:  ", block);
  for (int d = 0; d < BUCKETS; ++d) {
    std::printf(" %2u", counts[bucket_major_index(d, block, BLOCKS)]);
  }
  std::printf("\n");
}

static void radixPass(SortBuffers& buffers, int shift) {
  launch(radix_local_sort_kernel,
         dim3(BLOCKS),
         dim3(TILE),
         buffers.keys[0],
         N,
         shift,
         buffers.keysTmp,
         buffers.counts,
         buffers.localStart);
  launch(scan_table_kernel, dim3(1), dim3(TABLE_SIZE), buffers.counts, buffers.offsets, TABLE_SIZE);
  launch(radix_scatter_coalesced_kernel,
         dim3(BLOCKS),
         dim3(TILE),
         buffers.keysTmp,
         N,
         shift,
         buffers.offsets,
         buffers.localStart,
         buffers.keys[1]);
  CHECK(cudaGetLastError());
  std::swap(buffers.keys[0], buffers.keys[1]);
}

static void printPass(const SortBuffers& buffers, int shift) {
  std::printf("\ndigit of bits %d..%d\n", shift, shift + RADIX_BITS - 1);
  printKeys("tiles after local sort:", toHost(buffers.keysTmp, N));
  const std::vector<unsigned> counts = toHost(buffers.counts, TABLE_SIZE);
  for (int block = 0; block < BLOCKS; ++block) {
    printCounts(counts, block);
  }
  printKeys("after scatter:", toHost(buffers.keys[0], N));
}

static std::vector<unsigned> makeKeys() {
  std::vector<unsigned> keys(N);
  for (int i = 0; i < N; ++i) {
    keys[i] = (i % DISTINCT_KEYS * KEY_STEP + FIRST_KEY) % KEY_RANGE;
  }
  return keys;
}

int main() {
  const std::vector<unsigned> keys = makeKeys();
  const std::vector<unsigned> table(TABLE_SIZE, 0u);
  SortBuffers buffers = {{toDevice(keys), toDevice(keys)},
                         toDevice(keys),
                         toDevice(table),
                         toDevice(table),
                         toDevice(table)};

  printKeys("input (hex):", keys);
  for (int shift = 0; shift < KEY_BITS; shift += RADIX_BITS) {
    radixPass(buffers, shift);
    printPass(buffers, shift);
  }

  const std::vector<unsigned> sortedKeys = toHost(buffers.keys[0], N);
  unsigned* allBuffers[] = {buffers.keys[0],
                            buffers.keys[1],
                            buffers.keysTmp,
                            buffers.counts,
                            buffers.offsets,
                            buffers.localStart};
  for (unsigned* buffer : allBuffers) {
    CHECK(cudaFree(buffer));
  }

  std::vector<unsigned> expected = keys;
  std::sort(expected.begin(), expected.end());
  const bool ok = sortedKeys == expected;
  std::printf("\n%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_radix_multibit.cu -o 04_radix_multibit && ./04_radix_multibit

**What to look at in the output:** Read only the low hex digit of the keys after the first pass: 0 0 1 2 3 5 5 6 7 8 a a b b c d, so one pass sorted by 4 bits at once. Each counts row has 16 buckets for one block (in the second pass block 0 has 2 keys in bucket 9, the two 95s). The second pass sorts by the high digit and keeps the low-digit order inside each bucket, so 3a 3a come before 4b 4b and the 8-bit keys are sorted after 2 passes.

## Step 5 of 5: `05_merge_sort.cu`

Parallel merge sort: each thread sorts a run of 4 keys with insertion sort, then every pass merges pairs of runs with `co_rank`, each thread taking 3 outputs even when its slice crosses into the next pair.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_merge_sort.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <utility>
#include <vector>

constexpr int N = 16;
constexpr int RUN = 4;
constexpr int OUTPUTS_PER_THREAD = 3;
constexpr int THREADS = (N + OUTPUTS_PER_THREAD - 1) / OUTPUTS_PER_THREAD;
constexpr int RUNS = (N + RUN - 1) / RUN;
constexpr int LABEL_SIZE = 32;

__host__ __device__ int co_rank(int k, const unsigned* A, int m, const unsigned* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__host__ __device__ void merge_sequential(
    const unsigned* A, int m, const unsigned* B, int n, unsigned* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (A[i] <= B[j]) {
      C[k++] = A[i++];
    } else {
      C[k++] = B[j++];
    }
  }
  while (i < m) {
    C[k++] = A[i++];
  }
  while (j < n) {
    C[k++] = B[j++];
  }
}

__host__ __device__ void stable_insertion_sort(unsigned* keys, int begin, int end) {
  for (int i = begin + 1; i < end; ++i) {
    const unsigned key = keys[i];
    int j = i - 1;
    while (j >= begin && keys[j] > key) {
      keys[j + 1] = keys[j];
      --j;
    }
    keys[j + 1] = key;
  }
}

__global__ void sort_runs_kernel(unsigned* keys, int n, int run) {
  const int runStart = (blockIdx.x * blockDim.x + threadIdx.x) * run;
  if (runStart < n) {
    const int runEnd = imin(runStart + run, n);
    stable_insertion_sort(keys, runStart, runEnd);
  }
}

__global__ void merge_pass_kernel(
    const unsigned* keysIn, unsigned* keysOut, int total, int width, int outputsPerThread) {
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int sliceBegin = imin(tid * outputsPerThread, total);
  const int sliceEnd = imin(sliceBegin + outputsPerThread, total);

  int k = sliceBegin;
  while (k < sliceEnd) {
    const int pairStart = (k / (2 * width)) * (2 * width);
    const int mid = imin(pairStart + width, total);
    const int pairEnd = imin(pairStart + 2 * width, total);
    const int pieceEnd = imin(sliceEnd, pairEnd);

    const int m = mid - pairStart;
    const int n = pairEnd - mid;
    const int k_curr = k - pairStart;
    const int k_next = pieceEnd - pairStart;
    const int i_curr = co_rank(k_curr, keysIn + pairStart, m, keysIn + mid, n);
    const int i_next = co_rank(k_next, keysIn + pairStart, m, keysIn + mid, n);
    const int j_curr = k_curr - i_curr;
    const int j_next = k_next - i_next;
    merge_sequential(keysIn + pairStart + i_curr,
                     i_next - i_curr,
                     keysIn + mid + j_curr,
                     j_next - j_curr,
                     keysOut + k);

    k = pieceEnd;
  }
}

static unsigned* toDevice(const std::vector<unsigned>& values) {
  unsigned* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(unsigned)));
  CHECK(
      cudaMemcpy(pointer, values.data(), values.size() * sizeof(unsigned), cudaMemcpyHostToDevice));
  return pointer;
}

static std::vector<unsigned> toHost(const unsigned* pointer, int count) {
  std::vector<unsigned> values(count);
  CHECK(cudaMemcpy(values.data(), pointer, count * sizeof(unsigned), cudaMemcpyDeviceToHost));
  return values;
}

static void printRuns(const char* label, const std::vector<unsigned>& keys, int width) {
  std::printf("%-26s", label);
  for (int i = 0; i < N; ++i) {
    const bool startsRun = i > 0 && i % width == 0;
    std::printf("%s %2u", startsRun ? " |" : "", keys[i]);
  }
  std::printf("\n");
}

int main() {
  const std::vector<unsigned> keys = {9, 3, 14, 3, 7, 0, 12, 5, 9, 1, 15, 6, 2, 11, 7, 4};
  unsigned* keys_d[2] = {toDevice(keys), toDevice(keys)};

  printRuns("input:", keys, N);
  launch(sort_runs_kernel, dim3(1), dim3(RUNS), keys_d[0], N, RUN);
  CHECK(cudaGetLastError());
  printRuns("sorted runs of 4:", toHost(keys_d[0], N), RUN);

  for (int width = RUN; width < N; width *= 2) {
    launch(merge_pass_kernel,
           dim3(1),
           dim3(THREADS),
           keys_d[0],
           keys_d[1],
           N,
           width,
           OUTPUTS_PER_THREAD);
    CHECK(cudaGetLastError());
    std::swap(keys_d[0], keys_d[1]);
    char label[LABEL_SIZE];
    std::snprintf(label, sizeof label, "merged into runs of %d:", 2 * width);
    printRuns(label, toHost(keys_d[0], N), 2 * width);
  }

  const std::vector<unsigned> sortedKeys = toHost(keys_d[0], N);
  CHECK(cudaFree(keys_d[0]));
  CHECK(cudaFree(keys_d[1]));

  std::vector<unsigned> expected = keys;
  std::sort(expected.begin(), expected.end());
  const bool ok = sortedKeys == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_merge_sort.cu -o 05_merge_sort && ./05_merge_sort

**What to look at in the output:** Each vertical bar marks a run boundary: 4 sorted runs of 4 become 2 runs of 8 and then 1 run of 16, so 16 keys need 2 merge passes after the run sort. In the pass that builds runs of 8, 6 threads take 3 outputs each, and thread 2 owns outputs 6 to 8, so its slice crosses from the first pair of runs into the second and it calls `co_rank` for each piece. Try: set `OUTPUTS_PER_THREAD = 5`; 4 threads cover the 16 outputs and the result is the same.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/12_sorting/README.md`.

**Next:** open `colab_2_full_programs.ipynb` in this same folder.